In [1]:
import gymnasium as gym
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import trange

from sorbonne_rl_projects.ddpg import DDPG
from sorbonne_rl_projects.replay_buffer import ReplayBuffer
from sorbonne_rl_projects.td3 import TD3

%load_ext autoreload
%autoreload 2

Most of the following code is based on https://github.com/sfujim/td3

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def eval_env(policy: DDPG | TD3, env_name: str, eval_episodes: int = 10):
    eval_env = gym.make(env_name, continuous=True)

    avg_reward = 0.0
    for _ in range(eval_episodes):
        state, _ = eval_env.reset()
        terminated, truncated = False, False
        while not (terminated or truncated):
            action = policy.select_action(np.array(state))
            state, reward, terminated, truncated, _ = eval_env.step(action)
            avg_reward += reward

    avg_reward /= eval_episodes

    # print("---------------------------------------")
    print(f"Evaluation over {eval_episodes} episodes: {avg_reward:.3f}")
    # print("---------------------------------------")
    return avg_reward

In [ ]:
def calculate_overestimation_bias(
    policy: DDPG | TD3, env_name: str, eval_episodes: int = 10
):
    eval_env = gym.make(env_name, continuous=True)
    critic = policy.critic_1 if isinstance(policy, TD3) else policy.critic
    biases = []

    for _ in range(eval_episodes):
        q_values = []
        rewards = []

        state, _ = eval_env.reset()
        terminated, truncated = False, False
        while not (terminated or truncated):
            action = policy.select_action(np.array(state))
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).reshape(1, -1).to(policy.device)
                action_tensor = (
                    torch.FloatTensor(action).reshape(1, -1).to(policy.device)
                )
                q_values.append(critic(state_tensor, action_tensor).item())

            state, reward, terminated, truncated, _ = eval_env.step(action)
            rewards.append(reward)

        # Discounted Monte Carlo returns from each timestep.
        returns = []
        G = 0.0
        for reward in reversed(rewards):
            G = reward + policy.discount * G
            returns.append(G)
        returns.reverse()

        biases.extend(np.array(q_values) - np.array(returns))

    eval_env.close()

    return float(np.mean(biases))


In [ ]:
env_name: str = "LunarLander-v3"
seed: int = 1
max_timesteps = int(400e3)  # learns in about 300k steps
start_timesteps = int(25e3)
batch_size: int = 256
eval_freq: int = int(5e3)
expl_noise = 0.1


def run_policy(
    Policy: DDPG | TD3, env_name, seed: int = 1, layer_normalization: bool = False
):
    # Create environment
    env = gym.make(env_name, continuous=True)

    # Set seeds
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Get environment dimensions
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = float(env.action_space.high[0])

    # Create policy and replay buffer
    policy = Policy(state_dim, action_dim, max_action)
    replay_buffer = ReplayBuffer(state_dim, action_dim)

    # Use a separate TensorBoard run for each policy and seed.
    with SummaryWriter(
        comment=f"_{Policy.__name__}_seed_{seed}{'_LN' if layer_normalization else ''}"
    ) as writer:
        # Evaluate on
        evaluations = [eval_env(policy, env_name)]
        writer.add_scalar("reward/evaluation", evaluations[0], 0)

        state, _ = env.reset()
        terminated, truncated = False, False
        episode_reward = 0
        episode_timesteps = 0
        episode_num = 0

        progress_bar = trange(max_timesteps)
        for timestep in progress_bar:
            episode_timesteps += 1

            # Initially start with completely random actions
            if timestep < start_timesteps:
                action = env.action_space.sample()
            else:
                action = (
                    policy.select_action(np.asarray(state))
                    + np.random.normal(0, max_action * expl_noise, size=action_dim)
                ).clip(env.action_space.low, env.action_space.high)

            # Perform action
            next_state, reward, terminated, truncated, _ = env.step(action)
            writer.add_scalar("reward/step", reward, timestep + 1)

            # Store data in replay buffer
            replay_buffer.add(
                torch.tensor(state),
                torch.tensor(action),
                torch.tensor(next_state),
                torch.tensor(reward),
                torch.tensor(terminated),
                torch.tensor(truncated),
            )

            state = next_state
            episode_reward += reward

            # Train agent after collecting sufficient data
            if timestep >= start_timesteps:
                policy.train(replay_buffer, batch_size)

            if terminated or truncated:
                # +1 to account for 0 indexing. +0 on episode_timesteps since it will increment +1 even if terminated or truncated == True
                # print(
                #     f"Total T: {timestep + 1} Episode Num: {episode_num + 1} Episode T: {episode_timesteps} Reward: {episode_reward:.3f}"
                # )
                progress_bar.set_description(f"Reward: {episode_reward}")
                writer.add_scalar("reward/episode", episode_reward, timestep + 1)

                # Reset environment
                state, _ = env.reset()
                terminated, truncated = False, False
                episode_reward = 0
                episode_timesteps = 0
                episode_num += 1

            # Evaluate episode
            if (timestep + 1) % eval_freq == 0:
                evaluations.append(eval_env(policy, env_name))
                writer.add_scalar("reward/evaluation", evaluations[-1], timestep + 1)

                bias = calculate_overestimation_bias(policy, env_name)
                writer.add_scalar("critic/overestimation_bias", bias, timestep + 1)

                # np.save(f"./results/{file_name}", evaluations)
                # if args.save_model:
                #     policy.save(f"./models/{file_name}")

    env.close()

    return (policy, evaluations)

In [8]:
run_policy(DDPG, env_name, 1, layer_normalization=False)

Evaluation over 10 episodes: -152.715


  0%|          | 0/400000 [00:00<?, ?it/s]

Evaluation over 10 episodes: -143.663
Evaluation over 10 episodes: -149.838
Evaluation over 10 episodes: -135.958
Evaluation over 10 episodes: -158.888
Evaluation over 10 episodes: -146.580
Evaluation over 10 episodes: -202.578
Evaluation over 10 episodes: -139.299
Evaluation over 10 episodes: -54.980
Evaluation over 10 episodes: -13.642
Evaluation over 10 episodes: -9.971
Evaluation over 10 episodes: -69.807
Evaluation over 10 episodes: -53.867
Evaluation over 10 episodes: -143.220
Evaluation over 10 episodes: -144.168
Evaluation over 10 episodes: -121.903
Evaluation over 10 episodes: -106.060
Evaluation over 10 episodes: -35.468
Evaluation over 10 episodes: -19.407
Evaluation over 10 episodes: -42.354
Evaluation over 10 episodes: -14.909
Evaluation over 10 episodes: 18.865
Evaluation over 10 episodes: -25.226
Evaluation over 10 episodes: -4.226
Evaluation over 10 episodes: -38.248
Evaluation over 10 episodes: -31.544
Evaluation over 10 episodes: -30.895
Evaluation over 10 episodes: -

(<sorbonne_rl_projects.ddpg.DDPG at 0x113576120>,
 [np.float64(-152.71485534109925),
  np.float64(-143.66346961823226),
  np.float64(-149.837829385028),
  np.float64(-135.9579990554324),
  np.float64(-158.88774648262827),
  np.float64(-146.5798346705928),
  np.float64(-202.57781834157672),
  np.float64(-139.29946095034194),
  np.float64(-54.979765786657445),
  np.float64(-13.642304783695966),
  np.float64(-9.971249814747946),
  np.float64(-69.80693492860613),
  np.float64(-53.866953087837636),
  np.float64(-143.21972944762115),
  np.float64(-144.16844696656352),
  np.float64(-121.90281387424243),
  np.float64(-106.0595768173537),
  np.float64(-35.46773684582184),
  np.float64(-19.406852128220734),
  np.float64(-42.35376241928531),
  np.float64(-14.909282488916222),
  np.float64(18.86477487430911),
  np.float64(-25.22574953543869),
  np.float64(-4.226037883052237),
  np.float64(-38.24827763288185),
  np.float64(-31.543819924819367),
  np.float64(-30.895174012761366),
  np.float64(-11.62

In [9]:
run_policy(DDPG, env_name, 1, layer_normalization=True)

Evaluation over 10 episodes: -186.684


  0%|          | 0/400000 [00:00<?, ?it/s]

Evaluation over 10 episodes: -184.702
Evaluation over 10 episodes: -122.021
Evaluation over 10 episodes: -167.459
Evaluation over 10 episodes: -150.232
Evaluation over 10 episodes: -193.802
Evaluation over 10 episodes: -188.303
Evaluation over 10 episodes: -98.718
Evaluation over 10 episodes: -78.003
Evaluation over 10 episodes: -180.861
Evaluation over 10 episodes: -243.937
Evaluation over 10 episodes: -145.900
Evaluation over 10 episodes: -50.166
Evaluation over 10 episodes: -47.070
Evaluation over 10 episodes: -18.454
Evaluation over 10 episodes: -22.530
Evaluation over 10 episodes: -6.663
Evaluation over 10 episodes: -65.687
Evaluation over 10 episodes: -145.744
Evaluation over 10 episodes: -167.740
Evaluation over 10 episodes: -5.495
Evaluation over 10 episodes: 21.004
Evaluation over 10 episodes: 17.964
Evaluation over 10 episodes: -19.366
Evaluation over 10 episodes: -3.444
Evaluation over 10 episodes: 3.105
Evaluation over 10 episodes: -19.753
Evaluation over 10 episodes: -13.9

(<sorbonne_rl_projects.ddpg.DDPG at 0x117283750>,
 [np.float64(-186.68448129124278),
  np.float64(-184.70229394890393),
  np.float64(-122.02144442179747),
  np.float64(-167.45942096844092),
  np.float64(-150.23231197336364),
  np.float64(-193.8018003266421),
  np.float64(-188.30325646865654),
  np.float64(-98.7175071266551),
  np.float64(-78.00330528041724),
  np.float64(-180.8611339095579),
  np.float64(-243.93738722549278),
  np.float64(-145.89960094316731),
  np.float64(-50.166223213129584),
  np.float64(-47.0704832671557),
  np.float64(-18.454493263460154),
  np.float64(-22.52962895496905),
  np.float64(-6.663220373601658),
  np.float64(-65.68681025821019),
  np.float64(-145.7440749623112),
  np.float64(-167.73957778665508),
  np.float64(-5.4950774585076605),
  np.float64(21.003502873433664),
  np.float64(17.96444578078753),
  np.float64(-19.3661070906945),
  np.float64(-3.4442295024356895),
  np.float64(3.1048227428698754),
  np.float64(-19.753206665963994),
  np.float64(-13.94369

In [7]:
run_policy(TD3, env_name, 1)

Evaluation over 10 episodes: -100.590


  0%|          | 0/400000 [00:00<?, ?it/s]

Evaluation over 10 episodes: -145.670
Evaluation over 10 episodes: -147.143
Evaluation over 10 episodes: -137.126
Evaluation over 10 episodes: -112.542
Evaluation over 10 episodes: -128.495
Evaluation over 10 episodes: -127.345
Evaluation over 10 episodes: 2.205
Evaluation over 10 episodes: 40.548
Evaluation over 10 episodes: -16.350
Evaluation over 10 episodes: 34.714
Evaluation over 10 episodes: 57.495
Evaluation over 10 episodes: -34.829
Evaluation over 10 episodes: -18.244
Evaluation over 10 episodes: 0.638
Evaluation over 10 episodes: -45.916
Evaluation over 10 episodes: 121.664
Evaluation over 10 episodes: 97.347
Evaluation over 10 episodes: 68.544
Evaluation over 10 episodes: 11.369
Evaluation over 10 episodes: 76.705
Evaluation over 10 episodes: 68.142
Evaluation over 10 episodes: -15.721
Evaluation over 10 episodes: 131.788
Evaluation over 10 episodes: 12.539
Evaluation over 10 episodes: 26.340
Evaluation over 10 episodes: 48.388
Evaluation over 10 episodes: 0.336
Evaluation o

(<sorbonne_rl_projects.td3.TD3 at 0x1139552b0>,
 [np.float64(-100.58979464239243),
  np.float64(-145.66956921637552),
  np.float64(-147.14332361681255),
  np.float64(-137.12571397776838),
  np.float64(-112.54233157455685),
  np.float64(-128.49507208996542),
  np.float64(-127.34462183866984),
  np.float64(2.20485045032992),
  np.float64(40.54842606625759),
  np.float64(-16.349924756486764),
  np.float64(34.71438170266782),
  np.float64(57.49475943078555),
  np.float64(-34.82862858657491),
  np.float64(-18.243736366653177),
  np.float64(0.6379740205406701),
  np.float64(-45.91558878304859),
  np.float64(121.66416882675112),
  np.float64(97.34659339976568),
  np.float64(68.54407852842203),
  np.float64(11.36887863711792),
  np.float64(76.7051402536542),
  np.float64(68.14216642404509),
  np.float64(-15.720500290878045),
  np.float64(131.7876711791842),
  np.float64(12.539105038780017),
  np.float64(26.34043296116339),
  np.float64(48.38773994906019),
  np.float64(0.3355577135479531),
  np